# Variant and Gene Context


## How the candidate list was created

The source study performed whole-genome sequencing on heart tissue from people
with early-onset advanced heart failure. Sequencing produced millions of
variants. The researchers narrowed that list in several steps:

1. They focused on 369 genes connected to heart failure and related conditions.
2. A ranking pipeline used call quality, predicted consequence, splicing,
   conservation, population frequency, protein predictions, and ClinVar.
3. Variants with a study score of at least 15, plus previously reported
   likely pathogenic or pathogenic ClinVar variants, received manual review.
4. Clinical geneticists applied ACMG guidance and condition-specific rules.

Supplementary Table S4 contains the variants the authors reported as
pathogenic, likely pathogenic, or VUS with suggestive evidence. It has 54 rows
from 46 people and 25 genes.

The study score and class are starting information. This module will not
recalculate them. GTEx, HuBMAP, and Pharos answer different questions about the
genes connected to those variants.

## Inspect the Published Variants

Run the same steps used in the variant-background notebook.

First, load the complete Supplementary Table S4 teaching file and preview its
published fields.


In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
variants = pd.read_csv(DATA_DIR / "variants.csv")
variants.head()


Next, confirm the expected columns, row count, and paper identifier. These
checks prevent later joins from silently losing records.


In [ ]:
required_columns = {
    "subject_id",
    "gene_symbol",
    "hgvs_c",
    "hgvs_p",
    "study_pathogenicity_score",
    "study_class",
    "phenotype",
    "source_pmid",
}
missing_columns = required_columns.difference(variants.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert len(variants) == 54, "Expected all 54 rows from Supplementary Table S4"
assert variants["source_pmid"].eq(39910139).all()
print("Validated the complete published candidate table.")


Summarize the number of variant rows, people, genes, and missing protein HGVS
annotations without removing any records.


In [ ]:
dataset_overview = pd.Series(
    {
        "variant rows": len(variants),
        "subjects": variants["subject_id"].nunique(),
        "genes": variants["gene_symbol"].nunique(),
        "missing HGVSp values": variants["hgvs_p"].isna().sum(),
    },
    name="count",
).to_frame()
dataset_overview


Finally, count the published variant classes within each heart-failure
phenotype. The `(P)` label remains separate because the paper treated those
rows as pathogenic secondary findings.


In [ ]:
class_by_phenotype = (
    variants.groupby(["phenotype", "study_class"], dropna=False)
    .size()
    .rename("variant_rows")
    .reset_index()
    .sort_values(["phenotype", "study_class"])
)
class_by_phenotype
